In [27]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, when, round
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [7]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [19]:
# A. EXTRACT (bobot 15%)

# Baca ketiga sumber data (tugas6_transaksi.csv, tugas6_produk.json, tugas6_ulasan.csv) menjadi tiga Spark DataFrame terpisah. 
# Tampilkan jumlah baris dan printSchema() masing-masing.

df_transaksi = spark.read.csv(("tugas6_transaksi.csv"), header=True, inferSchema=True)
df_transaksi.printSchema()
df_transaksi.show()

df_produk = spark.read.json(("tugas6_produk.json"))
df_produk.printSchema()
df_produk.show()

df_ulasan = spark.read.csv(("tugas6_ulasan.csv"), header=True, inferSchema=True)
df_ulasan.printSchema()
df_ulasan.show()


root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

+--------+----------+------------+-------------------+
|order_id|product_id|unit_terjual|            tanggal|
+--------+----------+------------+-------------------+
|     TX0|        21|           2|2026-10-10 00:00:00|
|     TX1|         8|           6|2026-10-25 00:00:00|
|     TX2|        25|           4|2026-10-13 00:00:00|
|     TX3|         3|           4|2026-10-18 00:00:00|
|     TX4|        19|           6|2026-10-07 00:00:00|
|     TX5|        10|           3|2026-10-01 00:00:00|
|     TX6|        26|           6|2026-10-24 00:00:00|
|     TX7|         4|           1|2026-10-03 00:00:00|
|     TX8|         7|           2|2026-10-09 00:00:00|
|     TX9|        30|           1|2026-10-27 00:00:00|
|    TX10|        20|           1|2026-10-04 00:00:00|
|    TX11|        15|           4|2026-10-01 00:00:00|

In [20]:
# B. TRANSFORM — Penggabungan (bobot 25%)

# Gabungkan ketiga DataFrame menjadi satu (order_id sebagai kunci ke ulasan, product_id 
# sebagai kunci ke produk). Gunakan salah satu join yang tepat dari transaksi ke ulasan 
# (karena tidak semua transaksi memiliki ulasan) dan salah satu join yang tepat dari 
# transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). 
# Tambahkan kolom total_pendapatan (unit_terjual x harga).

df_gabung = df_transaksi.join(df_produk, on="product_id", how="inner") \
            .join(df_ulasan, on="order_id", how="left")

df_gabung = df_gabung.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))
df_gabung.show(5)



+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
o

In [24]:
# C. TRANSFORM — Penanganan Data Kosong & Pengayaan (bobot 20%)

# Transaksi tanpa ulasan akan memiliki rating 
# bernilai kosong (null) setelah salah satu join yang tepat
# — isi nilai kosong tersebut dengan angka 0 menggunakan salah satu function, 
# sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus 
# "belum ada ulasan".
# - Tambahkan kolom ada_ulasan bernilai True/False (tidak boleh diisi manual satu satu),
# **sebelum** langkah na.fill()` di atas).

df_rating = df_gabung.withColumn(
    "ada_ulasan",
    when(col("rating").isNotNull(), True).otherwise(False)
)

# 2. Isi nilai null pada kolom rating dengan angka 0
df_final_etl = df_enriched.na.fill({"rating": 0})

df_final_etl.select("order_id", "nama_produk", "kategori", "rating", "ada_ulasan", "total_pendapatan").show(10)

+--------+-----------+------------+------+----------+----------------+
|order_id|nama_produk|    kategori|rating|ada_ulasan|total_pendapatan|
+--------+-----------+------------+------+----------+----------------+
|     TX0|  Produk-21|  Elektronik|     1|      true|           50000|
|     TX1|   Produk-8|     Fashion|     0|     false|         1500000|
|     TX2|  Produk-25|  Elektronik|     2|      true|          100000|
|     TX3|   Produk-3|     Fashion|     5|      true|          300000|
|     TX4|  Produk-19|Rumah Tangga|     0|     false|          450000|
|     TX5|  Produk-10|  Elektronik|     5|      true|          225000|
|     TX6|  Produk-26|   Kesehatan|     4|      true|          450000|
|     TX7|   Produk-4|     Makanan|     0|     false|          150000|
|     TX8|   Produk-7|  Elektronik|     5|      true|          150000|
|     TX9|  Produk-30|  Elektronik|     0|     false|          250000|
+--------+-----------+------------+------+----------+----------------+
only s

In [25]:
# D. LOAD (bobot 25%)

# Simpan hasil akhir ke HDFS dalam format Parquet, dipartisi berdasarkan kategori, 
# ke path /user/[username]/tugas6/hasil_etl. Verifikasi dengan hdfs dfs -ls -R, 
# lalu baca kembali dan tampilkan count()-nya sebagai bukti data tersimpan utuh.

hdfs_path = "hdfs://localhost:9000/user/krock/tugas6/hasil_etl"

!hdfs dfs -mkdir -p /user/krock/tugas6

df_final_etl.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)

print("Verifikasi File di HDFS:")
!hdfs dfs -ls -R /user/krock/tugas6/hasil_etl

# Verifikasi dan tampilkan count
df_verifikasi = spark.read.parquet(hdfs_path)
print("\nJumlah baris terverifikasi di HDFS:", df_verifikasi.count())

Verifikasi File di HDFS:
-rw-r--r--   3 krock supergroup          0 2026-09-30 21:02 /user/krock/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - krock supergroup          0 2026-09-30 21:02 /user/krock/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 krock supergroup      16364 2026-09-30 21:02 /user/krock/tugas6/hasil_etl/kategori=Elektronik/part-00000-46022b78-b3ec-442e-8658-282aef23de75.c000.snappy.parquet
drwxr-xr-x   - krock supergroup          0 2026-09-30 21:02 /user/krock/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 krock supergroup      11134 2026-09-30 21:02 /user/krock/tugas6/hasil_etl/kategori=Fashion/part-00000-46022b78-b3ec-442e-8658-282aef23de75.c000.snappy.parquet
drwxr-xr-x   - krock supergroup          0 2026-09-30 21:02 /user/krock/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 krock supergroup      10393 2026-09-30 21:02 /user/krock/tugas6/hasil_etl/kategori=Kesehatan/part-00000-46022b78-b3ec-442e-8658-282aef23de75.c000.snappy.parquet
drwxr-xr-x   - krock superg

[Stage 48:>                                                         (0 + 3) / 3]


Jumlah baris terverifikasi di HDFS: 5000


In [28]:
# E. Insight Akhir (bobot 15%)

# Dari data hasil ETL, tampilkan (menggunakan DataFrame API atau Spark SQL, 
# bebas memilih): kategori produk mana yang memiliki persentase transaksi dengan
# ulasan (ada_ulasan = True) paling rendah? Tulis 2-3 kalimat interpretasi bisnis 
# pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?



df_insight = df_final_etl.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    count(when(col("ada_ulasan") == True, True)).alias("transaksi_berulasan")
).withColumn(
    "persentase_ulasan",
    round((col("transaksi_berulasan") / col("total_transaksi")) * 100, 2)
).orderBy("persentase_ulasan") # round disini digunakan untuk membulatkan nilai desimal

df_insight.show()

[Stage 53:>                                                         (0 + 1) / 1]

+------------+---------------+-------------------+-----------------+
|    kategori|total_transaksi|transaksi_berulasan|persentase_ulasan|
+------------+---------------+-------------------+-----------------+
|     Makanan|            536|                369|            68.84|
|   Kesehatan|            961|                662|            68.89|
|     Fashion|           1035|                726|            70.14|
|Rumah Tangga|            815|                575|            70.55|
|  Elektronik|           1653|               1168|            70.66|
+------------+---------------+-------------------+-----------------+



In [ ]:
Jawaban Interpretasi bisnis
-> mengapa melihat presentase transaksi dengan ulasan paling rendah ini termasuk hal 
penting diketahui oleh tim marketing, karena data tersebut mungkin dapat digunakan
untuk merencanakan strategi pemasaran. Semakin rendah presentase ulasan, maka
semakin rendah interaksi dari pembeli setelah menerima barang. Data ini sangat 
krusial mengingat tim Marketing juga harus merancang program insentif khusus agar 
mendapat ulasan lebih baik.

In [30]:
spark.stop()
print("SparkSession ditutup")

SparkSession ditutup
